# 9.1 ISC-CI Model

The ISC-CI model introducing a mechanism for context inference, based on the key assumption that temporal co-occurrence provides a useful basis for inferring shared context. Specifically, it assumes that (1) objects occurring together in a given context tend to share the properties elicited by that context; (2) these co-occurrence statistics are learned over the course of development; and (3) this implicit knowledge provides a basis for inferring, from a few examples of objects encountered in a new context, both which features are relevant in that context and what other objects are likely to occur in that context.

To make these ideas clear, consider the contexts in which you might encounter different kinds of birds: a bird-watching field trip in science class, a visit to the bird section of the zoo, and a picture book about birds. Each situation involves multiple types of birds (e.g., robins, crows, and ravens) and exposure to multiple bird-related properties (e.g., can-fly, eats-worms, is-bird) in various combinations. After these experiences, encountering a new context in which birds are relevant (e.g., learning that crows and ravens have hollow bones in the bird section of the Natural History museum) is likely to be interpreted as relating specifically to birds and their
properties, implying that other birds like robins may also occur in this new context, and that
they will share similar properties (e.g., robins also have hollow bones).

Conversely, contexts such as a science lesson on aerodynamics, a visit to a flight exhibit at a science museum, and
a film on the history of flight are likely to involve multiple types of flying things (e.g., crows,
airplanes, and butterflies) and flight-related properties (e.g., can-fly, has-wings, seen-in-the-sky). This suggests that a new context involving flying objects such as crows and airplanes (e.g., learning that crows and airplanes are associated with Bernoulli’s principle) likely relates to all things that can fly, implying that other flying things like butterflies may also occur in this new context and, again, share similar properties (e.g., butterflies are also associated with Bernoulli’s principle).

Thus, the properties shared by items encountered in a situation can provide a clue about what the current context is, what properties are currently important, and what other items are likely or unlikely also to be observed. The central hypothesis embodied by the ISC-CI model is that learning such environmental structure can support future inferences about which features might be relevant in novel contexts, based on the distribution of items that co-occur in those contexts. That is, observing that a new context involves a certain set of objects (e.g., both robins and airplanes) provides evidence that certain features will be context-relevant (e.g., can-fly and has-wings), but not
others (e.g., lays-eggs), based on past experience.

Importantly, this process is graded and probabilistic rather than absolute, as any given set of objects can co-occur in different contexts at different frequencies. In particular, features that are broadly true of many objects are less likely to be relevant in a new context than features that are true of the more limited set of objects seen in that context (Griffiths et al., 2010; Xu & Tenenbaum, 2007). This is because there is a low likelihood of observing any particular set of objects in a broad context: there are many animals, but few Corvidae, so it is more likely that a context involving both crows and ravens relates to Corvidae specifically than it is that this context relates to animals in general. This is because the probability of observing both crows and ravens in the context of animals is lower than the probability of oberseving both crows and ravens in the context of Corvidae.

*Setup and Installation:*

In [5]:
%%capture
%pip install psyneulink

import psyneulink as pnl
import pandas as pd

## Embedding

In [6]:
EMBEDDING_PATH = 'https://raw.githubusercontent.com/PrincetonUniversity/NEU-PSY-502/refs/heads/main/data/isc_ci/embeddings.csv'

embeddings_df = pd.read_csv(EMBEDDING_PATH, index_col=0)

chicken_embedding = embeddings_df.loc['chicken']
chicken_embedding

0     0.119731
1     0.987248
2     0.982023
3     0.410961
4     0.451753
        ...   
59    0.022675
60    0.326714
61    0.014851
62    0.035483
63    0.982558
Name: chicken, Length: 64, dtype: float64

## Generating the Training Data

We designed a training environment that simulates experiencing object co-occurrences throughout learning under the key assumption noted just above. The environment consisted of a series of episodes corresponding to different contexts. Each context involved a set of objects that share a common semantic feature (e.g., things that are birds, things that can fly, things that are found in the zoo, etc.), with each feature represented by a single output unit as
implemented by the feature labels in the ISC-CI model.

The model was trained on two objectives in each episode. First, it had to predict the feature label given the set of objects. For example, given the set {robin, canary}, the model should predict that is_a_bird is the semantic feature shared by items in the current context. Accordingly, we refer to this as the “bird” context. Second, the model had to predict which additional objects are likely to also occur in that context. For example, when in the “bird” context, the model should predict that sparrow is likely to occur but jaguar is not. We refer to the objects observed in the context (e.g., {robin, canary}) as the support set and the objects about which the model needs to make predictions in that context (e.g., {sparrow, jaguar}) as the query set (following terminology from metalearning; Thrun & Pratt, 1998).

We generated the episodes using the objects and features in the Leuven Concepts Database (De Deyne & Storms, 2008; Storms, 2001; Ruts et al., 2004). That database contains a matrix of binary judgments provided by human raters indicating, for each object-feature pairing, whether or not the object possess the feature (e.g., does a bear weigh more than 100lbs? Are kangaroos found in zoos?). After removing duplicate features and features that are only true of 2 items or fewer, the dataset contained 293 objects and 385 features.

We constructed a set of episodes by uniformly sampling from the set of features with replacement, so that each episode involved one shared semantic feature that defined the associated context. Given this feature, we generated a support set by uniformly sampling two items sharing the feature, and a query set by uniformly sampling one additional item sharing the feature (positive) and one that is not sharing the feature (negative). Note, that as above-mentioned, any given set of objects can co-occur in multiple contexts (in other words can share multiple features), however, for features tha are broadly true of many objects, it is less likely for each pair specific pair of objects to be sampled. For example, if the feature is "is_a_black_bird", the likelihood of sampling exactly {crow, raven} is higher than if the feature is "is_a_bird" (since there are many more birds than black birds).

For example, one episode consisted of the “zoo” context with the support set {zebra, elephant} and the query set {lion, rat}. In this episode the model had to first process {zebra, elephant} to infer that it was in the “zoo” context (ie, activating the zoo semantic feature as the important shared property of zebras and elephants in the current context). It then had to process the query set {lion, rat} to infer that lions occur in the context but rats do not. The model was trained in a supervised fashion to activate the important shared semantic
feature for support items encountered in the episode, and to generate a binary yes/no prediction for each object in the query set indicating whether or not that object belongs in the context. Importantly, each episode involved a single semantic feature shared by the support set and relevant to the inferred context that served as the target output for the model. That is,
even though the support set may have shared many features, only one of these was relevant to
the to-be-inferred context in a given episode. For instance, the support set {crow, robin} could
have been sampled from the “bird” context (i.e., is_a_bird as the context-relevant shared property) or the “animal” context (i.e., is_an_animal as the context-relevant shared property. If
occurring in the “bird” context, the semantic feature is_a_bird received a target of 1 and the
feature is_an_animal received a target of zero, and vice-versa if the same two items occurred in
the “animal” context. This ambiguity encouraged the model to learn a probability distribution
over the possible contexts that could be correct given the support set.

In [7]:
FEATURE_PATH = 'https://raw.githubusercontent.com/PrincetonUniversity/NEU-PSY-502/refs/heads/main/data/isc_ci/features.csv'

feature_df = pd.read_csv(FEATURE_PATH, index_col=0)

feature_df.head()

,is small,is a bird,is an animal,is big,can fly,is an insect,mammal,is a fish,lays eggs,is brown,...,is for all ages,you can play different notes whit it,can be used to put something in,can be bought in sports store,costs a lot of money,drives above the ground,driven by 1 person,used in water,used in the house,worn often
monkey,0,0,1,0,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
beaver,0,0,1,0,0,0,1,0,0,1,...,0,0,0,0,0,0,0,0,0,0
bison,0,0,1,1,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
dromedary,0,0,1,1,0,0,1,0,0,1,...,0,0,0,0,0,0,0,0,0,0
squirrel,1,0,1,0,0,0,1,0,0,1,...,0,0,0,0,0,0,0,0,0,0


First, let's create a function that randomly picks a context, then randomly picks two supports for this context and either a query that is or is not in the context:

In [43]:
import random

def get_example():
    # randomly pick a context (feature)
    feature = random.choice(feature_df.columns)
    # one hot encoded feature vector is our context:
    feature_vector = [0] * len(feature_df.columns)
    feature_vector[feature_df.columns.get_loc(feature)] = 1

    # get only the objects that have this feature by name
    objects_included = feature_df[feature_df[feature] == 1].index.tolist()
    objects_excluded = feature_df[feature_df[feature] == 0].index.tolist()

    # randomly pick two supports (can be the same twice)
    support = random.choices(objects_included, k=2)

    # the support vector is the embedding of the two supports
    support_vector_1 = embeddings_df.loc[support[0]]
    support_vector_2 = embeddings_df.loc[support[1]]

    # decide weather to pick an object that is in the context or not
    choice = random.choice([0, 1])

    if choice == 0:
        # pick an object that is not in the context
        query = random.choice(objects_excluded)
    else:
        # pick an object that is not in the context
        query = random.choice(objects_included)

    # the query vector is the embedding of the query
    query_vector = embeddings_df.loc[query]
    print(query_vector)

    # get only the objects




get_example()

KeyError: 'double bass'

## PsyNeuLink Model

In [ ]:
N_EMBEDDING = embeddings_df.shape[1]
HIDDEN_CONTEXT = 100
N_CONTEXT = 100

def create_model(
):
    support_1 = pnl.TransferMechanism(
        name='support_1',
        input_shapes=N_EMBEDDING
    )

    support_2 = pnl.TransferMechanism(
        name='support_2',
        input_shapes=N_EMBEDDING
    )

    mean_support = pnl.TransferMechanism(
        name='mean_support',
        input_shapes=N_EMBEDDING,
        function=pnl.Linear(slope=.5),

    )

    context_dependent = pnl.TransferMechanism(
        name='context_dependent',
        input_shapes=HIDDEN_CONTEXT,
        function=pnl.ReLU()
    )

    context_out = pnl.TransferMechanism(
        name='context_out',
        input_shapes=N_CONTEXT,
        function=pnl.SoftMax()
    )

    query_x = pnl.TransferMechanism(
        name='query_x',
        input_shapes=N_EMBEDDING
    )

    query_dependent = pnl.TransferMechanism(
        name='query_dependent',
        input_shapes=N_CONTEXT,
        function=pnl.ReLU()
    )

    response = pnl.TransferMechanism(
        name='response',
        input_shapes=1,
        function=pnl.Logistic(),
    )

    projection_mean_to_context_dependent = pnl.MappingProjection(
        name='projection_mean_to_context_dependent',
        matrix=pnl.RandomMatrix(),
        learnable=True
    )

    projection_query_to_query_dependent = pnl.MappingProjection(
        name='projection_query_to_query_dependent',
        matrix=pnl.RandomMatrix(),
        learnable=True
    )

    projection_context_dependent_to_context_out = pnl.MappingProjection(
        name='projection_context_dependent_to_context_out',
        matrix=pnl.RandomMatrix(),
        learnable=True
    )

    projection_query_dependent_to_response = pnl.MappingProjection(
        name='projection_query_dependent_to_response',
        matrix=pnl.RandomMatrix(),
        learnable=True
    )

    model = pnl.Composition(
        name='embedding',
        pathways=[
            [support_1, pnl.IDENTITY_MATRIX, mean_support],
            [support_2, pnl.IDENTITY_MATRIX, mean_support],
            [query_x, projection_query_to_query_dependent, query_dependent],
            [mean_support, projection_mean_to_context_dependent, context_dependent],
            [context_dependent, pnl.IDENTITY_MATRIX, query_dependent],
            [context_dependent, projection_context_dependent_to_context_out, context_out],
            [query_dependent, projection_query_dependent_to_response, response]
        ]
    )
    return (model, support_1, support_2,
            context_out, response)

In [ ]:
(isc_ci, support_1, support_2, context_out, response) = create_model()
isc_ci.show_graph(output_fmt='jupyter')